# Stage 6 (Updated): Feature Importance — 10-Stock, 50 Features (MDI / MDA / SFI)

Runs three AFML feature-importance methods on the pooled 10-stock modelling dataset
(2,071 events × 50 features: 17 TS time-series + 33 WorldQuant 101 alphas).

- **MDI**: mean decrease in impurity from a fitted RF (fast, slight high-bias)
- **MDA**: permutation importance via `MultiAssetPurgedKFold` (weighted neg-log-loss)
- **SFI**: single-feature purged CV accuracy (weighted)

Key question: *do any of the 33 alpha features rank in the top-10 of any method?*

In [ ]:
import sys, os, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

sys.path.insert(0, os.path.abspath('../..'))
from src.cross_validation import MultiAssetPurgedKFold, PurgedKFold
from src.modelling import train_and_evaluate
from src.feature_importance import (
    feat_imp_MDI, feat_imp_MDA, feat_imp_SFI, plot_feature_importance,
)

plt.style.use('seaborn-v0_8-darkgrid')
RNG = 42
os.makedirs('../../reports/figures', exist_ok=True)

## 1. Load pooled dataset and baseline RF params

In [ ]:
dataset = pd.read_parquet('../../data/processed/pooled_modelling.parquet')

meta_cols  = {'label', 't1', 'weight', 'ticker'}
ts_cols    = [c for c in dataset.columns if c not in meta_cols and not c.startswith('alpha')]
alpha_cols = [c for c in dataset.columns if c.startswith('alpha')]
feat_cols  = ts_cols + alpha_cols

X = dataset[feat_cols]
y = dataset['label'].astype(int)
sample_weight = dataset['weight']
t1 = dataset['t1']

# Baseline RF: balanced, shallow (no tuning yet)
rf_params = {
    'n_estimators': 200, 'max_depth': 4, 'min_samples_leaf': 5,
    'class_weight': 'balanced', 'max_features': 'sqrt',
}

majority_baseline = float((y == y.mode()[0]).mean())
print(f'Dataset shape     : {X.shape}  (TS={len(ts_cols)}, alpha={len(alpha_cols)})')
print(f'majority baseline : {majority_baseline:.4f}')

pkf = MultiAssetPurgedKFold(n_splits=5, t1=t1, pct_embargo=0.01)
print('CV: MultiAssetPurgedKFold(n_splits=5, pct_embargo=0.01)')

## 2. Fit tuned RF on full data

In [ ]:
rf = RandomForestClassifier(**rf_params, random_state=RNG, n_jobs=-1)
rf.fit(X, y, sample_weight=sample_weight.values)
rf

## 3. MDI

In [ ]:
mdi = feat_imp_MDI(rf, X.columns); mdi.round(4)

In [ ]:
fig = plot_feature_importance(mdi, 'MDI (mean decrease in impurity)', color='#2c7fb8')
fig.savefig('../../reports/figures/pooled/P15_mdi_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. MDA — weighted neg_log_loss permutation

Both the baseline and per-feature permuted scores honour the test-fold sample weights, so the MDA delta is computed on a like-for-like basis.

In [ ]:
mda = feat_imp_MDA(
    clf=rf, X=X, y=y, cv=pkf,
    sample_weight=sample_weight, scoring='neg_log_loss', random_state=RNG,
)
mda.round(4)

In [ ]:
fig = plot_feature_importance(mda, 'MDA (Δ neg log-loss under permutation, weighted)',
                              color='#e34a33')
fig.savefig('../../reports/figures/pooled/P16_mda_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. SFI — weighted single-feature CV

In [ ]:
sfi_template = RandomForestClassifier(**rf_params, random_state=RNG, n_jobs=-1)
sfi = feat_imp_SFI(
    clf_template=sfi_template, X=X, y=y, cv=pkf,
    sample_weight=sample_weight, scoring='neg_log_loss',
)
sfi.round(4)

In [ ]:
fig = plot_feature_importance(sfi, 'SFI (single-feature weighted CV neg log-loss)',
                              color='#31a354')
fig.savefig('../../reports/figures/pooled/P17_sfi_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Rank comparison and Spearman correlation

In [ ]:
rank_df = pd.DataFrame({
    'MDI_rank': mdi['mean'].rank(ascending=False).astype(int),
    'MDA_rank': mda['mean'].rank(ascending=False).astype(int),
    'SFI_rank': sfi['mean'].rank(ascending=False).astype(int),
}).reindex(X.columns)
rank_df['avg_rank'] = rank_df.mean(axis=1)
rank_df = rank_df.sort_values('avg_rank')
rank_df

In [ ]:
rank_corr = rank_df[['MDI_rank', 'MDA_rank', 'SFI_rank']].corr(method='spearman')
fig, ax = plt.subplots(figsize=(4.5, 3.5))
im = ax.imshow(rank_corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(rank_corr.columns); ax.set_yticklabels(rank_corr.index)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{rank_corr.values[i, j]:.2f}', ha='center', va='center', fontsize=10)
ax.set_title('Spearman rank correlation (MDI / MDA / SFI)')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

## 7. Bottom-25% intersection rule

Drop a feature only if it is in the bottom 25 % on **all three** rankings.

In [ ]:
n_features = len(X.columns)
bottom_n = max(1, int(np.ceil(n_features * 0.25)))
bottom_threshold = n_features - bottom_n + 1
is_bottom = (rank_df[['MDI_rank', 'MDA_rank', 'SFI_rank']] >= bottom_threshold)
drop_features = list(rank_df.index[is_bottom.all(axis=1)])
keep_features = [c for c in X.columns if c not in drop_features]
print(f'features dropped ({len(drop_features)}): {drop_features}')
print(f'features kept    ({len(keep_features)}): {keep_features}')

## 8. Refit reduced model

In [ ]:
X_red = X[keep_features]
rf_full_template = RandomForestClassifier(**rf_params, random_state=RNG, n_jobs=-1)
rf_red_template  = RandomForestClassifier(**rf_params, random_state=RNG, n_jobs=-1)

result_full = train_and_evaluate(rf_full_template, X,     y, sample_weight,
                                 cv=PurgedKFold(5, t1=t1, pct_embargo=0.01),
                                 scoring='accuracy')
result_red  = train_and_evaluate(rf_red_template,  X_red, y, sample_weight,
                                 cv=PurgedKFold(5, t1=t1, pct_embargo=0.01),
                                 scoring='accuracy')

compare = pd.DataFrame({
    'mean_acc':   [result_full['mean_score'], result_red['mean_score']],
    'std_acc':    [result_full['std_score'],  result_red['std_score']],
    'n_features': [X.shape[1], X_red.shape[1]],
    'beats_baseline': [result_full['mean_score'] > majority_baseline,
                      result_red['mean_score']  > majority_baseline],
}, index=['full', 'reduced']).round(4)
compare

## 9. Persist artefacts

In [ ]:
feature_importance = pd.concat({'MDI': mdi, 'MDA': mda, 'SFI': sfi}, axis=1)
feature_importance.columns = ['_'.join(c) for c in feature_importance.columns]
feature_importance['MDI_rank'] = rank_df['MDI_rank']
feature_importance['MDA_rank'] = rank_df['MDA_rank']
feature_importance['SFI_rank'] = rank_df['SFI_rank']
feature_importance['avg_rank'] = rank_df['avg_rank']
feature_importance['kept']     = ~feature_importance.index.isin(drop_features)
feature_importance = feature_importance.reindex(X.columns)
feature_importance.to_parquet('../../data/processed/feature_importance.parquet')

with open('../../models/model_final.pkl', 'wb') as f:
    pickle.dump(result_red['fitted_clf'], f)

print('Saved:')
print('  ../data/processed/feature_importance.parquet')
print('  ../models/model_final.pkl')
print('  ../reports/figures/P15_mdi_importance.png')
print('  ../reports/figures/P16_mda_importance.png')
print('  ../reports/figures/P17_sfi_importance.png')

## 10. Stage 4 / 5 / 6 comparison summary

Single table loading the regenerated artefacts: majority baseline, Stage 4 baseline RF / XGB, Stage 5 tuned RF / XGB, and Stage 6 reduced RF.

In [ ]:
import json
with open('../../models/best_params_pooled.json') as f:
    best = json.load(f)

cv_results = pd.read_parquet('../../data/processed/cv_results.parquet')
rf_baseline_mean  = float(cv_results.loc['mean', 'RF'])
xgb_baseline_mean = float(cv_results.loc['mean', 'XGB'])
rf_baseline_std   = float(cv_results.loc['std',  'RF'])
xgb_baseline_std  = float(cv_results.loc['std',  'XGB'])

rf_tuned_mean  = best['rf']['mean_score']
rf_tuned_std   = best['rf']['std_score']
xgb_tuned_mean = best['xgb']['mean_score']
xgb_tuned_std  = best['xgb']['std_score']

summary = pd.DataFrame([
    {'row': 'majority-class baseline',     'mean_acc': majority_baseline, 'std_acc': np.nan,
     'beats_baseline': False, 'note': 'always predict majority class'},
    {'row': 'Stage 4 RF (untuned)',        'mean_acc': rf_baseline_mean,  'std_acc': rf_baseline_std,
     'beats_baseline': rf_baseline_mean  > majority_baseline,
     'note': 'AFML-recipe params, weighted purged CV'},
    {'row': 'Stage 4 XGB (untuned)',       'mean_acc': xgb_baseline_mean, 'std_acc': xgb_baseline_std,
     'beats_baseline': xgb_baseline_mean > majority_baseline,
     'note': 'AFML-recipe params, weighted purged CV'},
    {'row': 'Stage 5 RF (tuned)',          'mean_acc': rf_tuned_mean,     'std_acc': rf_tuned_std,
     'beats_baseline': rf_tuned_mean    > majority_baseline,
     'note': '25-iter randomised search, weighted purged CV'},
    {'row': 'Stage 5 XGB (tuned)',         'mean_acc': xgb_tuned_mean,    'std_acc': xgb_tuned_std,
     'beats_baseline': xgb_tuned_mean   > majority_baseline,
     'note': '25-iter randomised search, weighted purged CV'},
    {'row': 'Stage 6 RF (reduced)',        'mean_acc': result_red['mean_score'],
     'std_acc': result_red['std_score'],
     'beats_baseline': result_red['mean_score'] > majority_baseline,
     'note': f"tuned RF on {len(keep_features)} features after pruning"},
]).set_index('row')
summary.round(4)

## 11. Notes

- All accuracy numbers in this notebook are **weighted** under the   AFML uniqueness-decay scheme (both fitting and scoring).
- The DSR-on-CV-trials reported in Stage 5 corrects only for   in-sample selection bias across hyperparameter configurations.   A *strategy* DSR over realised PnL belongs to the backtesting   stage, not here.
- If the bottom-25% intersection turns out empty, the reduced   model equals the full model (as we report explicitly above).